In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy

In [ ]:
from datasets import load_dataset

dataset = load_dataset("liamdugan/raid", split="train")
print(dataset)

In [ ]:
import pandas as pd

# Load more rows to get enough human samples
df = dataset.select(range(400000)).to_pandas()

del dataset
import gc
gc.collect()

print(df.shape)
print(df['model'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
import gc

df = df[['generation', 'model', 'domain']].dropna()
df = df.rename(columns={'generation': 'text'})
df['label'] = df['model'].apply(lambda x: 0 if x == 'human' else 1)

# Filter only abstracts domain for better real world testing
df = df[df['domain'] == 'abstracts']

human_count = len(df[df['label'] == 0])
ai_count = len(df[df['label'] == 1])
print(f"Human: {human_count}, AI: {ai_count}")

sample_size = min(human_count, ai_count, 19000)
print(f"Sampling: {sample_size} each")

ai = df[df['label'] == 1].sample(sample_size, random_state=42)
human = df[df['label'] == 0].sample(sample_size, random_state=42)
df_balanced = pd.concat([ai, human]).sample(frac=1, random_state=42).reset_index(drop=True)

del df
gc.collect()

train_df, test_df = train_test_split(df_balanced, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

print(f"Total balanced: {len(df_balanced)}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np

# ── 1. Build vocabulary from training data ──────────────────────────────────
def build_vocab(texts, max_vocab=30000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.most_common(max_vocab):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df['text'])
print(f"Vocabulary size: {len(vocab)}")

# ── 2. Tokenizer for LSTM (integer indices, no subword splitting) ────────────
def tokenize(texts, vocab, max_len=128):
    input_ids      = []
    attention_mask = []

    for text in texts:
        tokens = text.lower().split()[:max_len]
        ids    = [vocab.get(t, vocab['<UNK>']) for t in tokens]

        pad_len = max_len - len(ids)
        mask    = [1] * len(ids) + [0] * pad_len
        ids     = ids             + [vocab['<PAD>']] * pad_len

        input_ids.append(ids)
        attention_mask.append(mask)

    return {
        'input_ids':      torch.tensor(input_ids,      dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
    }

train_encodings = tokenize(train_df['text'], vocab)
val_encodings   = tokenize(val_df['text'],   vocab)
test_encodings  = tokenize(test_df['text'],  vocab)

print("Tokenization complete!")

# ── 3. LSTM model (drop-in replacement for BERT) ────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 num_layers=2, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, num_classes)  # ×2 for bidirectional

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)                      # (B, L, E)
        x, (hidden, _) = self.lstm(x)                      # (B, L, H*2)

        # Use the final forward + backward hidden states
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (B, H*2)
        out    = self.fc(self.dropout(hidden))               # (B, num_classes)
        return out

model = LSTMClassifier(vocab_size=len(vocab))
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_encodings, list(train_df['label']))
val_dataset   = TextDataset(val_encodings,   list(val_df['label']))
test_dataset  = TextDataset(test_encodings,  list(test_df['label']))

print("Dataset created!")

In [ ]:
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels=2, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0, bidirectional=True)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim * 2, num_labels))
        self.loss_fn    = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.embedding(input_ids)
        if attention_mask is not None:
            x = nn.utils.rnn.pack_padded_sequence(x, attention_mask.sum(1).cpu(),
                                                   batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(x)
        logits = self.classifier(torch.cat([hidden[-2], hidden[-1]], dim=1))
        loss   = self.loss_fn(logits, labels) if labels is not None else None
        return loss, logits

model = LSTMClassifier(vocab_size=len(vocab), num_labels=2)
print(f"Model loaded! | Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    load_best_model_at_end=True,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print(classification_report(labels, preds, target_names=['Human', 'AI']))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/lstm_mgd_model.pt'

torch.save({
    'model_state': model.state_dict(),
    'vocab':       vocab,
    'config':      {'vocab_size': len(vocab), 'num_labels': 2, 'embed_dim': 128,
                    'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.3}
}, SAVE_PATH)

print(f"Model saved! → {SAVE_PATH}")

In [ ]:
def predict_text(text):
    # Use the custom tokenize function, which expects a list of texts
    tokenized_output = tokenize([text], vocab, max_len=128)

    # Move tensors to the correct device
    input_ids = tokenized_output['input_ids'].to('cuda' if torch.cuda.is_available() else 'cpu')
    attention_mask = tokenized_output['attention_mask'].to('cuda' if torch.cuda.is_available() else 'cpu')

    # Prepare inputs dictionary for the model
    inputs = {'input_ids': input_ids, 'attention_mask': attention_mask}

    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    with torch.no_grad():
        # The model's forward method returns a tuple (loss, logits)
        _, logits = model(**inputs)
        prediction = torch.argmax(logits, dim=1).item()
        confidence = torch.softmax(logits, dim=1).max().item()

    label = "🤖 AI Generated" if prediction == 1 else "👤 Human Written"
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence*100:.2f}%")

# Interactive input
while True:
    text = input("\nEnter text to check (or type 'quit' to stop): ")
    if text.lower() == 'quit':
        print("Exiting...")
        break
    predict_text(text)